In [6]:
import json
import os
from datetime import datetime
from pathlib import Path
import pandas as pd

# 1. Автоматическое определение корня проекта
current_dir = Path.cwd()
project_root = current_dir.parent if current_dir.name == "notebooks" else current_dir
raw_dir = project_root / "data" / "raw" / "variant_02"

# 2. Поиск сырых JSON файлов
raw_files = sorted(list(raw_dir.glob("*variant_02*.json"))) or sorted(list(raw_dir.glob("*.json")))

if not raw_files:
    raise FileNotFoundError(
        f"Файлы JSON не найдены по пути: {raw_dir.resolve()}\n"
        f"Убедимся, что в консоли был выполнен запуск: python src/extract.py --config configs/variant_02.yml"
    )

latest_raw_file = raw_files[-1]
print(f"[INFO] Загрузка файла: {latest_raw_file}")

with open(latest_raw_file, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

# 3. Извлечение почасовых данных и создание DataFrame
hourly_raw = raw_data.get("hourly", {})
df = pd.DataFrame(hourly_raw)

print(f"[OK] DataFrame успешно создан. Строк: {df.shape[0]}, Столбцов: {df.shape[1]}")
df.head()


[INFO] Загрузка файла: C:\Users\User\3D Objects\TP-end_to_end_project-main\data\raw\variant_02\2026-09-15_09-31-13.json
[OK] DataFrame успешно создан. Строк: 168, Столбцов: 5


,time,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m
0,2026-09-15T00:00,13.1,79,0.0,7.6
1,2026-09-15T01:00,12.4,82,0.0,6.1
2,2026-09-15T02:00,11.8,82,0.0,5.8
3,2026-09-15T03:00,11.1,81,0.0,5.4
4,2026-09-15T04:00,10.8,80,0.0,4.7


3. Разворачивание JSON в DataFrame
В Open-Meteo структура представляет собой словарь со списками равной длины


In [2]:
df = pd.DataFrame(hourly_raw)
print(f"Размер исходной таблицы: {df.shape}")
print(df.head())



Размер исходной таблицы: (168, 5)
               time  temperature_2m  relative_humidity_2m  precipitation  \
0  2026-09-15T00:00            13.1                    79            0.0   
1  2026-09-15T01:00            12.4                    82            0.0   
2  2026-09-15T02:00            11.8                    82            0.0   
3  2026-09-15T03:00            11.1                    81            0.0   
4  2026-09-15T04:00            10.8                    80            0.0   

   wind_speed_10m  
0             7.6  
1             6.1  
2             5.8  
3             5.4  
4             4.7  


==========================================
4. Осознанные шаги очистки и нормализации
==========================================


Шаг Очистки 1: Парсинг дат и временных меток
Зачем: По умолчанию временя читается как строка (object). Перевод в datetime 
позволяет строить временные ряды, делать группировки по дням и проверять порядок.


In [3]:
df["ts"] = pd.to_datetime(df["time"])
df.drop(columns=["time"], inplace=True)



Шаг Очистки 2: Явное приведение типов и добавление идентификатора бизнес-сущности
Зачем: Добавляем city_id для возможности объединения с другими городами и гарантируем float64


In [4]:
df["city_id"] = "RU_LED"
numeric_cols = ["temperature_2m", "relative_humidity_2m", "precipitation", "wind_speed_10m"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").astype(float)



Шаг Очистки 3: Удаление дубликатов по ключу сущности и сортировка
Зачем: Гарантирует уникальность записей по (city_id, ts) согласно Data Quality правилам


In [5]:
rows_before = len(df)
df = df.drop_duplicates(subset=["city_id", "ts"]).sort_values("ts").reset_index(drop=True)
rows_after = len(df)
print(f"Строк до очистки: {rows_before}, после удаления дубликатов: {rows_after}")

# Упорядочивание колонок согласно схеме
target_schema = ["ts", "temperature_2m", "relative_humidity_2m", "precipitation", "wind_speed_10m", "city_id"]
df = df[target_schema]

# 5. Проверка "здоровья" очищенной таблицы
print("\n=== Здоровье нормализованной таблицы ===")
print("df.head():\n", df.head(3))
print("\ndf.dtypes:\n", df.dtypes)
print("\nКоличество пропусков:\n", df.isna().sum())

# 6. Сохранение нормализованных данных в CSV
output_dir = Path("../data/normalized/variant_02")
output_dir.mkdir(parents=True, exist_ok=True)

timestamp_str = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
output_path = output_dir / f"{timestamp_str}.csv"

df.to_csv(output_path, index=False, encoding="utf-8")
print(f"\n[OK] Нормализованные данные сохранены в: {output_path}")


Строк до очистки: 168, после удаления дубликатов: 168

=== Здоровье нормализованной таблицы ===
df.head():
                    ts  temperature_2m  relative_humidity_2m  precipitation  \
0 2026-09-15 00:00:00            13.1                  79.0            0.0   
1 2026-09-15 01:00:00            12.4                  82.0            0.0   
2 2026-09-15 02:00:00            11.8                  82.0            0.0   

   wind_speed_10m city_id  
0             7.6  RU_LED  
1             6.1  RU_LED  
2             5.8  RU_LED  

df.dtypes:
 ts                      datetime64[us]
temperature_2m                 float64
relative_humidity_2m           float64
precipitation                  float64
wind_speed_10m                 float64
city_id                            str
dtype: object

Количество пропусков:
 ts                      0
temperature_2m          0
relative_humidity_2m    0
precipitation           0
wind_speed_10m          0
city_id                 0
dtype: int64

[OK] Нормали